---
title: "`pytask` Config: Defining the Pipeline Internals in `pytask`"
---

## config

> This is the config module for the `pytask` pipeline. 
This module defines the data catalog(s) and any hard-coded parameters that are used throughout the pipeline.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## `DEV_MODE`: A Quick Development Flag

I'm adding a flag to the config that can be used for quick development. 
If you import this boolean variable, it can be used to skip tasks,
setup samples, etc. on the fly by `marking` a task with the `pytask.mark.skipif`
decorator. Change this to `False` when you're ready to run the full pipeline.

## The Data Catalog

To manage our pipeline, we're going to use a nested data catalog structure.
This way, we can easily return specific entries to specific tasks
without having to manage multiple different data catalogs. Specifically,
we'll have a data catalog for each stage of the pipeline, and each catalog
will have entries for the inputs, outputs, and any other parameters needed
for that stage. This is similar to how we used Hydra configs, but
using the `pytask` data catalog, we can more easily gather the data
for a specific task in structured manner entirely in Python.

## The Download Task

A good strategy may be to set pipeline stage parameters in the config file, 
and then use the `pytask` data catalog to manage the data. This way, we can
easily change the parameters without having to modify the code. This is especially 
useful for the API query, where we need to be able to set the parameter grid for
the years and data types we want to download data for. So, let's create an entry in the data catalog specifically for the download task.

A good strategy I thought about for grid parameter comprehension is to create a dataframe expands all the combinations of
parameters, and then uses each combination to create the tasks which are then 
easily added to the data catalog. This way, we can still easily inspect the 
pipeline and see what tasks are being run, while also being able to easily 
change the parameters in the config file without too much hassle.

An important framework decision I'm making here is that each ROW of the dataframe corresponds to a single task, so that we can quickly understand at a glance what the task is doing, and also easily develop the code for the task itself. This is different from the hydra approach where a job is first specified by a default config, and then the parameters are swept over in multiple config files. This is a more flexible approach, IMO, because:

1. each row defines a single task run, so it's easy to understand what the run is doing
2. it's easy to add or remove runs by simply expanding the list of parameters and using dataframe filters to remove irrelevant parameter combinations
3. we don't have to independently inspect and manage multiple different/overriding config files
4. it's all in Python, so we can use the full power of the language to define
   the parameters and the tasks in a single sweep, not through the need of
   hydra+snakemake multi stage/multi-lingual config system

So, to do this, we define one job as a query to the CDS API that must contain:
- The dataset (re-analysis)
- The year
- The month
- All days in the month
- All times of day (hour)
- The geography (region), which will need:
    - The URL to the shapefile to calculate the bounding box

Given one combination of all of these, a single SLURM job can complete the first "task" in parallel by having a run assigned to each row of the dataframe.

In [ ]:
query_df

,year,month,geography,shapefile,product_type,day,time,variables,output
0,2009,01,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_01_madagascar
1,2009,01,nepal,https://data.humdata.org/dataset/07db728a-4f0f...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_01_nepal
2,2009,02,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_02_madagascar
3,2009,02,nepal,https://data.humdata.org/dataset/07db728a-4f0f...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_02_nepal
4,2009,03,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_03_madagascar
...,...,...,...,...,...,...,...,...,...
379,2024,10,nepal,https://data.humdata.org/dataset/07db728a-4f0f...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2024_10_nepal
380,2024,11,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2024_11_madagascar
381,2024,11,nepal,https://data.humdata.org/dataset/07db728a-4f0f...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2024_11_nepal
382,2024,12,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2024_12_madagascar


In [ ]:
print(f"Number of estimated jobs: {query_df.shape[0]}. Examples...")

for i, row in query_df.sample(3).iterrows():
    print(f"Year: {row['year']}, Month: {row['month']}, Geography: {row['geography']}, Link: {row['shapefile']}, Variables: {row['variables']}")

Number of estimated jobs: 384. Examples...
Year: 2016, Month: 11, Geography: madagascar, Link: https://data.humdata.org/dataset/26fa506b-0727-4d9d-a590-d2abee21ee22/resource/ed94d52e-349e-41be-80cb-62dc0435bd34/download/mdg_adm_bngrc_ocha_20181031_shp.zip, Variables: ['2m_dewpoint_temperature', '2m_temperature', 'total_precipitation', 'volumetric_soil_water_layer_1']
Year: 2009, Month: 11, Geography: madagascar, Link: https://data.humdata.org/dataset/26fa506b-0727-4d9d-a590-d2abee21ee22/resource/ed94d52e-349e-41be-80cb-62dc0435bd34/download/mdg_adm_bngrc_ocha_20181031_shp.zip, Variables: ['2m_dewpoint_temperature', '2m_temperature', 'total_precipitation', 'volumetric_soil_water_layer_1']
Year: 2018, Month: 04, Geography: nepal, Link: https://data.humdata.org/dataset/07db728a-4f0f-4e98-8eb0-8fa9df61f01c/resource/2eb4c47f-fd6e-425d-b623-d35be1a7640e/download/npl_adm_nd_20240314_ab_shp.zip, Variables: ['2m_dewpoint_temperature', '2m_temperature', 'total_precipitation', 'volumetric_soil_wa

Now add them to the catalog. We're going to use a dictionary to
nest data catalogs so that we can return specific task products to
named data catalog nodes.

Our data catalog now has a `download|jobs` node with a `queries_df` entry that contains the dataframe of all the jobs to be run in this task.

In [ ]:
data_catalog['download']['jobs']['queries_df'].load().head()

,year,month,geography,shapefile,product_type,day,time,variables,output
0,2009,01,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_01_madagascar
1,2009,01,nepal,https://data.humdata.org/dataset/07db728a-4f0f...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_01_nepal
2,2009,02,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_02_madagascar
3,2009,02,nepal,https://data.humdata.org/dataset/07db728a-4f0f...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_02_nepal
4,2009,03,madagascar,https://data.humdata.org/dataset/26fa506b-0727...,reanalysis,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1...","[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...","[2m_dewpoint_temperature, 2m_temperature, tota...",2009_03_madagascar


## The Aggregation Task

To carry out the aggregation, we will follow similar logic to the original pipeline and use xarray to aggregate data into spatial and temporal averages. The aggregation task will take the downloaded data and compute the mean over the specified time period and spatial region. However, in this case, we want to aggregate the data diurnally, so we will need to fetch the sundown and sunrise times for the region and use them to compute the diurnal averages.

Once again, we will use a dataframe to define the parameters for the aggregation task.

Here we will use a dataframe with the jobs as rows;
the first column is "input" which is the list of query names from
the download task, and the last column is the output object name. Columns
in between can be the parameters needed for the aggregation task, which
then get expanded to the full list of jobs with `itertools.product`, `explode` or similar,
and filtered as necessary.

For explanations of the parameters, see the Aggregation Task notebook's final `task_aggregate_data_diurnal` function.

Inspecting it:

In [ ]:
agg_params

,time,solar_classification,variables,variables_short,aggregation_name
0,day,before,2m_dewpoint_temperature,d2m,mean
1,day,before,2m_dewpoint_temperature,d2m,sum
2,day,before,2m_dewpoint_temperature,d2m,max
3,day,before,2m_dewpoint_temperature,d2m,min
4,day,before,2m_dewpoint_temperature,t2m,mean
...,...,...,...,...,...
123,night,before,volumetric_soil_water_layer_1,tp,min
124,night,before,volumetric_soil_water_layer_1,swvl1,mean
125,night,before,volumetric_soil_water_layer_1,swvl1,sum
126,night,before,volumetric_soil_water_layer_1,swvl1,max


Let's keep only rows where the variables and variables_short match

In [ ]:
agg_params

,time,solar_classification,variables,variables_short,aggregation_name
0,day,before,2m_dewpoint_temperature,d2m,mean
1,day,before,2m_dewpoint_temperature,d2m,sum
2,day,before,2m_dewpoint_temperature,d2m,max
3,day,before,2m_dewpoint_temperature,d2m,min
20,day,before,2m_temperature,t2m,mean
21,day,before,2m_temperature,t2m,sum
22,day,before,2m_temperature,t2m,max
23,day,before,2m_temperature,t2m,min
40,day,before,total_precipitation,tp,mean
41,day,before,total_precipitation,tp,sum


Great, and now keeping `sum` only for total precipitation (we don't need mean, max, min for that variable), and removing `sum` for all other variables (we don't need sum for temperature or soil moisture):

In [ ]:
agg_params

,time,solar_classification,variables,variables_short,aggregation_name
0,day,before,2m_dewpoint_temperature,d2m,mean
2,day,before,2m_dewpoint_temperature,d2m,max
3,day,before,2m_dewpoint_temperature,d2m,min
20,day,before,2m_temperature,t2m,mean
22,day,before,2m_temperature,t2m,max
23,day,before,2m_temperature,t2m,min
41,day,before,total_precipitation,tp,sum
60,day,before,volumetric_soil_water_layer_1,swvl1,mean
62,day,before,volumetric_soil_water_layer_1,swvl1,max
63,day,before,volumetric_soil_water_layer_1,swvl1,min


Now we add the input and output columns by joining:

This result gives us the full list of jobs for the aggregation task. 20 rows for the parameters,
and 384 inputs/outputs, giving a total of 7680 jobs:

In [ ]:
assert aggregate_jobs.shape[0] == 20 * len(inputs)
aggregate_jobs

,input,time,solar_classification,variables,variables_short,aggregation_name
0,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,mean
1,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,max
2,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,min
3,2009_01_madagascar,day,before,2m_temperature,t2m,mean
4,2009_01_madagascar,day,before,2m_temperature,t2m,max
...,...,...,...,...,...,...
7675,2024_12_nepal,night,before,2m_temperature,t2m,min
7676,2024_12_nepal,night,before,total_precipitation,tp,sum
7677,2024_12_nepal,night,before,volumetric_soil_water_layer_1,swvl1,mean
7678,2024_12_nepal,night,before,volumetric_soil_water_layer_1,swvl1,max


A few more configuration items need to be added, like
the local timezone for each geography, the healthshed filename,
the healthshed unique ID variable name in the shapefile,
and whether the variable is instantaneous or accumulated:

In [ ]:
aggregate_jobs

,input,time,solar_classification,variables,variables_short,aggregation_name,local_tz,shapefile,hshd_unique_id,climate_handler_var
0,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,mean,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
1,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,max,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
2,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,min,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
3,2009_01_madagascar,day,before,2m_temperature,t2m,mean,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
4,2009_01_madagascar,day,before,2m_temperature,t2m,max,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
...,...,...,...,...,...,...,...,...,...,...
7675,2024_12_nepal,night,before,2m_temperature,t2m,min,Asia/Kathmandu,Nepal_Healthsheds2024.zip,fid,instant
7676,2024_12_nepal,night,before,total_precipitation,tp,sum,Asia/Kathmandu,Nepal_Healthsheds2024.zip,fid,accum
7677,2024_12_nepal,night,before,volumetric_soil_water_layer_1,swvl1,mean,Asia/Kathmandu,Nepal_Healthsheds2024.zip,fid,instant
7678,2024_12_nepal,night,before,volumetric_soil_water_layer_1,swvl1,max,Asia/Kathmandu,Nepal_Healthsheds2024.zip,fid,instant


Now we add this to the data catalog:

Our data catalog now has an `aggregate|jobs` node with a `jobs_df` entry that contains the dataframe of all the jobs to be run in this task.

In [ ]:
data_catalog['aggregate']['jobs']['jobs_df'].load().head()

,input,time,solar_classification,variables,variables_short,aggregation_name,local_tz,shapefile,hshd_unique_id,climate_handler_var
0,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,mean,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
1,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,max,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
2,2009_01_madagascar,day,before,2m_dewpoint_temperature,d2m,min,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
3,2009_01_madagascar,day,before,2m_temperature,t2m,mean,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
4,2009_01_madagascar,day,before,2m_temperature,t2m,max,Indian/Antananarivo,healthsheds2022.zip,fs_uid,instant
